---
title: "Numeryczne rozwiązanie równania Burgersa metodą MPDATA"
subtitle: "Wpływ liczby Reynoldsa, rozdzielczości siatki i parametru CFL"
author: ""
date: ""
lang: pl
format:
  pdf:
    documentclass: article
    papersize: a4
    geometry:
      - top=0.7cm
      - bottom=0.7cm
      - left=0.9cm
      - right=0.9cm
    fontsize: 8pt
    fig-format: pdf
    colorlinks: false
    number-sections: false
    fig-pos: "H"
    include-in-header:
      text: |
        \usepackage{float}
        \usepackage{titlesec}
        \titlespacing*{\section}{0pt}{0.3em}{0.1em}
        \titlespacing*{\subsection}{0pt}{0.2em}{0.05em}
        \setlength{\parskip}{1pt}
        \setlength{\parindent}{0pt}
        \setlength{\textfloatsep}{2pt}
        \setlength{\intextsep}{2pt}
        \setlength{\abovecaptionskip}{1pt}
        \setlength{\belowcaptionskip}{0pt}
        \setlength{\abovedisplayskip}{3pt}
        \setlength{\belowdisplayskip}{3pt}
execute:
  echo: false
  warning: false
  message: false
---

# Model i metoda numeryczna

Rozważamy jednowymiarowe równanie Burgersa:

$$
\frac{\partial u}{\partial t}
+\frac{\partial}{\partial x}\left(\frac{u^2}{2}\right)
=\nu\frac{\partial^2u}{\partial x^2},
\qquad \nu=\frac{1}{Re}.
$$

Dla $u_L=1$, $u_R=0$, $x_0=0$ i prędkości fali $s=0{,}5$ rozwiązanie analityczne wynosi:

$$
u(x,t)=\frac12-\frac12\tanh\left(\frac{x-0{,}5t}{4\nu}\right).
$$

Symulacje wykonano metodą MPDATA (biblioteka **PyMPDATA**) na przedziale $[-10,10]$ z periodycznymi warunkami brzegowymi. Błąd liczono w obszarze $[-2,4]$, obejmującym falę i pomijającym wpływ granic domeny.

# Porównanie profili

Badano $Re\in\{25,50,100,200\}$. Profile porównano dla 8 komórek na szerokość fali i CFL $=0{,}05$. Szerokość fali wynosi:

$$
\delta=\frac{4\nu}{u_L-u_R}=\frac{4}{Re}.
$$


In [10]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.patches import Patch
from PyMPDATA import Options, ScalarField, VectorField, Stepper, Solver
from PyMPDATA.boundary_conditions import Periodic
from matplotlib_inline.backend_inline import set_matplotlib_formats
from matplotlib_inline.backend_inline import InlineBackend

InlineBackend.instance().figure_formats = {'pdf'}


# Parametry fizyczne i obliczeniowe
u_L, u_R, x_0 = 1.0, 0.0, 0.0
L = 20.0
x_min, x_max = -L/2, L/2
t_max = 2.0
error_region = (-2.0, 4.0)

options = Options(
    nonoscillatory=True,
    infinite_gauge=True,
    n_iters=3,
    non_zero_mu_coeff=True
)
boundary_conditions = (Periodic(),)

# Wspólne funkcje solvera MPDATA
def analytical_solution_Re(x, t, Re):
    nu = 1.0 / Re
    s = (u_L + u_R) / 2
    return (
        (u_L + u_R) / 2
        - (u_L - u_R) / 2
        * np.tanh((u_L-u_R)*(x-x_0-s*t)/(4*nu))
    )

def interpolate_advector(solver, dt, dx):
    u = solver.advectee.get()
    a = solver.advector.get_component(0)
    a[1:-1] = 0.25 * dt / dx * (u[1:] + u[:-1])

def extrapolate_advector(vectors_in, solver):
    a_out = solver.advector.get_component(0)
    a_out[1:-1] = 0.5 * (3*vectors_in[0] - vectors_in[1])

def check_cfl_condition(solver):
    cfl = np.max(np.abs(solver.advector.get_component(0)))
    if cfl > 1:
        raise RuntimeError(f"Warunek CFL naruszony: {cfl:.4g}")
    return cfl

def calculate_errors(u_num, u_exact):
    diff = u_num - u_exact
    return (np.mean(np.abs(diff)),
            np.sqrt(np.mean(diff**2)),
            np.max(np.abs(diff)))


Wielkość CFL określa, jaką część komórki siatki pokonuje informacja numeryczna w jednym kroku czasu. W eksperymencie parametr docelowy wyznacza krok czasu z maksymalnej prędkości adwekcji $|u|/2$. Liczba komórek na szerokość fali opisuje przestrzenną rozdzielczość profilu: większa wartość oznacza gęstszą siatkę.


In [2]:
# Dana konfiguracja: siatka, krok czasu, ewolucja i miary błędu
def test_configuration(Re, cells_per_shock, CFL, save_profile=False):
    nu = 1.0 / Re
    shock_width = 4*nu/(u_L-u_R)
    dx_target = shock_width/cells_per_shock
    Nx = int(np.ceil(L/dx_target))
    dx = L/Nx
    x = x_min + (np.arange(Nx)+0.5)*dx

    max_speed = max(abs(u_L), abs(u_R))/2
    dt = CFL*dx/max_speed
    steps = int(np.ceil(t_max/dt))
    dt = t_max/steps

    u_initial = analytical_solution_Re(x, 0.0, Re)
    solver = Solver(
        stepper=Stepper(options=options, grid=(Nx,)),
        advectee=ScalarField(data=u_initial.copy(), halo=options.n_halo,
                             boundary_conditions=boundary_conditions),
        advector=VectorField(data=(np.zeros(Nx+1),), halo=options.n_halo,
                             boundary_conditions=boundary_conditions)
    )
    buffers = [np.empty(Nx-1), np.empty(Nx-1)]
    interpolate_advector(solver, dt, dx)
    buffers[1][:] = solver.advector.get_component(0)[1:-1]
    mu_coeff = (nu*dt/dx**2,)
    stable, max_CFL = True, 0.0
    start = time.perf_counter()

    try:
        for step in range(1, steps+1):
            interpolate_advector(solver, dt, dx)
            buffers[0][:] = solver.advector.get_component(0)[1:-1]
            extrapolate_advector(buffers, solver)
            max_CFL = max(max_CFL, check_cfl_condition(solver))
            solver.advance(n_steps=1, mu_coeff=mu_coeff)
            buffers = buffers[::-1]
            if not np.all(np.isfinite(solver.advectee.get())):
                raise FloatingPointError(f"NaN/Inf po kroku {step}")
        u_num = solver.advectee.get().copy()
    except Exception:
        stable = False
        u_num = np.full(Nx, np.nan)

    runtime = time.perf_counter()-start
    mask = (x >= error_region[0]) & (x <= error_region[1])
    if stable:
        mae, rmse, max_error = calculate_errors(
            u_num[mask], analytical_solution_Re(x[mask], t_max, Re))
    else:
        mae = rmse = max_error = np.nan

    result = {
        'Re': Re, 'cells_per_shock': cells_per_shock, 'CFL_target': CFL,
        'Nx': Nx, 'dx': dx, 'dt': dt, 'steps': steps,
        'mu_coeff': mu_coeff[0], 'max_CFL': max_CFL,
        'stable': stable, 'MAE': mae, 'RMSE': rmse,
        'MaxError': max_error, 'runtime': runtime
    }
    if save_profile:
        result['_profile'] = (x, u_num)
    return result


# Eksperyment parametryczny

Przebadano 64 konfiguracje, łącząc cztery wartości $Re$, cztery rozdzielczości (4–32 komórek na szerokość fali) i cztery wartości CFL (0,025–0,2). Dla każdej oceniano stabilność, błąd rozwiązania i czas obliczeń. Konfiguracje niestabilne pominięto w analizie błędu i froncie Pareto.

In [3]:
#| output: false
Re_values = [25, 50, 100, 200]
cells_values = [4, 8, 16, 32]
CFL_values = [0.025, 0.05, 0.1, 0.2]

all_results = []
profile_results = {}
total = len(Re_values)*len(cells_values)*len(CFL_values)
count = 0
for Re in Re_values:
    for cells in cells_values:
        for CFL in CFL_values:
            count += 1
            keep_profile = (cells == 8 and CFL == 0.05)
            result = test_configuration(Re, cells, CFL, save_profile=keep_profile)
            if keep_profile and result['stable']:
                profile_results[Re] = result.pop('_profile')
            else:
                result.pop('_profile', None)
            all_results.append(result)
            print(f"{count:02d}/{total}: Re={Re}, komórki={cells}, CFL={CFL:g}, "
                  f"stabilna={result['stable']}, RMSE={result['RMSE']:.3e}")

df_results = pd.DataFrame(all_results)
df_valid = df_results[df_results['stable'] & np.isfinite(df_results['RMSE'])].copy()
assert len(df_results) == 64


01/64: Re=25, komórki=4, CFL=0.025, stabilna=True, RMSE=1.272e-03
02/64: Re=25, komórki=4, CFL=0.05, stabilna=True, RMSE=1.020e-03
03/64: Re=25, komórki=4, CFL=0.1, stabilna=True, RMSE=5.276e-04
04/64: Re=25, komórki=4, CFL=0.2, stabilna=False, RMSE=nan
05/64: Re=25, komórki=8, CFL=0.025, stabilna=True, RMSE=2.705e-04
06/64: Re=25, komórki=8, CFL=0.05, stabilna=True, RMSE=1.702e-04
07/64: Re=25, komórki=8, CFL=0.1, stabilna=False, RMSE=nan
08/64: Re=25, komórki=8, CFL=0.2, stabilna=False, RMSE=nan
09/64: Re=25, komórki=16, CFL=0.025, stabilna=True, RMSE=4.721e-05
10/64: Re=25, komórki=16, CFL=0.05, stabilna=False, RMSE=nan
11/64: Re=25, komórki=16, CFL=0.1, stabilna=False, RMSE=nan
12/64: Re=25, komórki=16, CFL=0.2, stabilna=False, RMSE=nan
13/64: Re=25, komórki=32, CFL=0.025, stabilna=False, RMSE=nan
14/64: Re=25, komórki=32, CFL=0.05, stabilna=False, RMSE=nan
15/64: Re=25, komórki=32, CFL=0.1, stabilna=False, RMSE=nan
16/64: Re=25, komórki=32, CFL=0.2, stabilna=False, RMSE=nan
17/64:

In [7]:
fig, axes = plt.subplots(2, 2, figsize=(7.0, 4.4), sharex=True, sharey=True)
for ax, Re in zip(axes.flat, Re_values):
    if Re not in profile_results:
        ax.text(0.5, 0.5, 'Brak stabilnego wyniku', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'Re = {Re}')
        continue
    x, u_num = profile_results[Re]
    mask = (x >= -1.0) & (x <= 3.0)
    ax.plot(x[mask], u_num[mask], label='MPDATA', lw=1.5)
    ax.plot(x[mask], analytical_solution_Re(x[mask], t_max, Re), '--', label='Analityczne', lw=1.2)
    ax.set_title(f'Re = {Re}')
    ax.grid(alpha=0.25)
    ax.set_xlabel('x')
    ax.set_ylabel('u(x, t = 2)')
axes[0,0].legend(fontsize=7, loc='best')
fig.tight_layout()
plt.show()

<Figure size 700x440 with 4 Axes>

# Stabilność i dokładność

Mapa przedstawia logarytm błędu RMSE dla poszczególnych konfiguracji; krzyżykiem oznaczono przypadki niestabilne. Błąd liczono w przedziale $[-2,4]$, z pominięciem obszarów przy granicach domeny.

In [8]:
fig, axes = plt.subplots(2, 2, figsize=(7.0, 5.0), sharex=True, sharey=True)

valid_rmse = df_valid.loc[df_valid['RMSE'] > 0, 'RMSE']
vmin, vmax = valid_rmse.min(), valid_rmse.max()

# Granice komórek heatmapy
X = np.arange(len(CFL_values) + 1)
Y = np.arange(len(cells_values) + 1)

for ax, Re in zip(axes.flat, Re_values):
    sub = df_results[df_results['Re'] == Re]

    pivot = sub.pivot(
        index='cells_per_shock',
        columns='CFL_target',
        values='RMSE'
    )

    stable = sub.pivot(
        index='cells_per_shock',
        columns='CFL_target',
        values='stable'
    )

    data = pivot.to_numpy(dtype=float)
    data[~stable.to_numpy(dtype=bool)] = np.nan

    mesh = ax.pcolormesh(
        X, Y, data,
        norm=LogNorm(vmin=vmin, vmax=vmax),
        cmap='viridis',
        shading='flat'
    )

    # Etykiety osi na środku komórek
    ax.set_xticks(np.arange(len(CFL_values)) + 0.5)
    ax.set_xticklabels([f'{v:g}' for v in CFL_values])

    ax.set_yticks(np.arange(len(cells_values)) + 0.5)
    ax.set_yticklabels([str(v) for v in cells_values])

    ax.set_title(f'Re = {Re}', fontsize=8)
    ax.set_xlabel('Docelowy CFL', fontsize=8)
    ax.set_ylabel('Komórki na szerokość fali')

    # Wartości RMSE wewnątrz komórek
    for i, cells in enumerate(cells_values):
        for j, CFL in enumerate(CFL_values):
            row = sub[
                (sub.cells_per_shock == cells) &
                (sub.CFL_target == CFL)
            ].iloc[0]

            label = (
                f"{row.RMSE:.1e}"
                if row.stable and np.isfinite(row.RMSE)
                else '×'
            )

            ax.text(
                j + 0.5, i + 0.5, label,
                ha='center', va='center',
                fontsize=7,
                color='white' if row.stable else 'black',
                fontweight='bold'
            )

fig.colorbar(
    mesh,
    ax=axes.ravel().tolist(),
    label='RMSE (skala logarytmiczna)',
    shrink=0.82
)

fig.suptitle('Błąd i stabilność konfiguracji', y=1.01)
plt.show()

<Figure size 700x500 with 5 Axes>

## Kompromis: dokładność a koszt obliczeń

Front Pareto wyznaczono osobno dla każdego $Re$. Obejmuje on stabilne konfiguracje niezdominowane pod względem RMSE i czasu obliczeń, czyli takie, których nie można poprawić jednocześnie w obu kryteriach.

In [9]:
pareto_rows = []
for Re in Re_values:
    sub = df_valid[df_valid['Re'] == Re]
    for _, candidate in sub.iterrows():
        dominated = ((sub.RMSE <= candidate.RMSE) &
                     (sub.runtime <= candidate.runtime) &
                     ((sub.RMSE < candidate.RMSE) | (sub.runtime < candidate.runtime))).any()
        if not dominated:
            pareto_rows.append(candidate)
df_pareto = pd.DataFrame(pareto_rows)

fig, axes = plt.subplots(2, 2, figsize=(7.0, 4.8), sharey=True)
for ax, Re in zip(axes.flat, Re_values):
    sub = df_valid[df_valid.Re == Re]
    front = df_pareto[df_pareto.Re == Re].sort_values('runtime')
    ax.scatter(sub.runtime, sub.RMSE, s=18, alpha=0.45, label='Stabilne konfiguracje')
    ax.scatter(front.runtime, front.RMSE, s=42, marker='D', edgecolor='black', linewidth=0.5,
               label='Front Pareto', zorder=3)
    if len(front) > 1:
        ax.plot(front.runtime, front.RMSE, '--', lw=0.9)
    ax.set_yscale('log')
    ax.set_title(f'Re = {Re}')
    ax.set_xlabel('Czas obliczeń [s]')
    ax.set_ylabel('RMSE')
    ax.grid(True, which='both', alpha=0.25)
axes[0,0].legend(fontsize=7)
fig.tight_layout()
plt.show()


<Figure size 700x480 with 4 Axes>